# 00 — 베이스 모델 선정 (스펙 §2 · **W1 하네스 교정**)

> ## ⚙️ 실행 모드 배너 — **Azure A100 80GB(japaneast, Spot)에서 실제 실행**된 zero-shot/2-shot 선정.
> `bench_baseselect.py`(정식 chat template · `enable_thinking=false` · 64-tok · 공식 KorQuAD F1 ·
> held-out 800)로 ≤9B 3종을 비교 → 결과 `results/base_select.json`. 이 노트북은 그 실측을 로드/표시.

### 이 단계 — 무엇/왜 (W1 교정)
- **문제(v1)**: zero-shot이 **하네스 아티팩트**였다 — raw 텍스트 템플릿(chat template 미적용) + 32-토큰 예산 →
  Qwen3(think 모드)이 잘림/잡음으로 **F1 3.5** 같은 비현실 점수. 모델 문제가 아니라 **측정 문제**.
- **교정(v2)**: 각 모델 **chat template** 사용 + Qwen3는 `enable_thinking=False`, **충분한 64-토큰**, 정답 어구만
  추출, **공식 KorQuAD EM/F1**, held-out 800(학습과 동일 seed), **zero-shot + 2-shot**(few-shot은 TRAIN에서 추출, 누출 없음).
- **후보 ≤9B 3종**: Qwen3-8B / Qwen2.5-7B-Instruct / Llama-3.1-8B(게이팅 시 ungated **Yi-1.5-9B-Chat** 자동 폴백).

### 0) 부트스트랩 & 설정

In [1]:
import os, sys, json, glob
here = os.getcwd()
for cand in [here, os.path.dirname(here), os.path.join(here, "pdf_qa_extraction"),
             os.path.dirname(os.path.dirname(here))]:
    if os.path.isdir(os.path.join(cand, "quantization")):
        if cand not in sys.path: sys.path.insert(0, cand)
        os.chdir(cand); break
print("cwd:", os.getcwd())

cwd: /home/azureuser/work/pdf_qa_extraction


In [2]:
from quantization.data_korquad import load_config
import quantization.v2_pipeline as V
cfg = load_config()
SEED = 42  # representative seed for the demo; metrics below aggregate all seeds
BASE = cfg["base_model"]["selected"]
print("base:", BASE, "| seeds:", cfg.get("seeds"), "| eval held-out:", cfg["data"]["eval_size"])

base: Qwen/Qwen3-8B | seeds: [42, 43, 44] | eval held-out: 1000


In [3]:
import platform, torch, transformers, torchao
env = {"python": platform.python_version(), "torch": torch.__version__,
       "transformers": transformers.__version__, "torchao": torchao.__version__,
       "cuda": torch.version.cuda,
       "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")}
os.makedirs("quantization/results", exist_ok=True)
json.dump(env, open("quantization/results/env_baseselect.json", "w"), ensure_ascii=False, indent=2)
env

/home/azureuser/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.10.12',
 'torch': '2.11.0+cu130',
 'transformers': '4.57.6',
 'torchao': '0.17.0',
 'cuda': '13.0',
 'device': 'NVIDIA A100 80GB PCIe'}

### 1) 후보 비교 (실측 zero-shot / 2-shot · EM·F1)

In [4]:
bs = json.load(open("quantization/results/base_select.json"))
print("%-24s %-17s %-17s %s" % ("candidate", "zero-shot EM/F1", "2-shot EM/F1", "note"))
for e in bs:
    z, f = e["zeroshot"], e["fewshot"]
    g = e.get("gated_fallback_from", "")
    note = ("fallback<-" + g.split("/")[-1]) if g else ""
    print("%-24s %5.1f/%-10.2f %5.1f/%-10.2f %s" % (
        e["candidate"].split("/")[-1], z["exact_match"], z["f1"],
        f["exact_match"], f["f1"], note))

candidate                zero-shot EM/F1   2-shot EM/F1      note
Qwen3-8B                  81.8/92.51       83.8/93.67      
Qwen2.5-7B-Instruct       76.9/88.90       79.4/90.07      
Yi-1.5-9B-Chat            47.8/73.46        0.2/9.68       fallback<-Llama-3.1-8B-Instruct


### 2) 동작 데모 — 승자의 실측 샘플 예측(gold/pred)

In [5]:
win = max(bs, key=lambda e: e["zeroshot"]["f1"])
print("[승자]", win["candidate"], "zero-shot F1=%.2f" % win["zeroshot"]["f1"])
print("샘플 예측(실측 gold/pred):")
for s in win["zeroshot"]["samples"]:
    print("  Q:", s["q"]) ; print("    gold:", s["gold"], "| pred:", s["pred"])

[승자] Qwen/Qwen3-8B zero-shot F1=92.51
샘플 예측(실측 gold/pred):
  Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
    gold: 대중교통체계 | pred: 대중교통체계
  Q: 11월 24일 김영삼이 대통령 명령으로 제정한 법은?
    gold: 5·18 관련 특별법 | pred: 특별법
  Q: 포켓몬스터 금은을 닌텐도 DS용으로 리메크한 것이 일본에서 발매된 해는?
    gold: 2009년 | pred: 2009


### 3) 선정 결정
**Qwen3-8B** = ungated 후보 중 zero-shot F1 최고(92.5, Qwen2.5-7B 88.9 · Yi 73.5) + 단일 A100 QAT 적합 +
TorchAO INT4/vLLM 호환. `config.yaml`의 `base_model.selected`에 고정 → A/B/C 동일 베이스.
> v1의 F1 3.5는 순수 하네스 아티팩트였음이 재확인됨(동일 모델이 정상 하네스에선 90+).

In [6]:
print("config base_model.selected =", cfg["base_model"]["selected"])
assert win["candidate"] == cfg["base_model"]["selected"], "winner != config base"
print("=> 확정: Qwen3-8B (ungated · 단일 A100 QAT 적합 · INT4/vLLM 호환)")

config base_model.selected = Qwen/Qwen3-8B
=> 확정: Qwen3-8B (ungated · 단일 A100 QAT 적합 · INT4/vLLM 호환)
